# 01 — Camada de dados (DAL)

Desenvolve as funções de acesso a dados (baixar, gravar/ler SQLite, retornos). **F1, NF6.**

In [2]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import sqlite3
from contextlib import closing
from typing import Sequence
import pandas as pd

## Desenvolvimento

A(s) função(ões)/classe(s) abaixo foi(ram) escrita(s) aqui e, após os testes, movida(s) para `app/dal.py`.

In [3]:
def _validar_identificador(nome: str) -> None:
    """Impede nomes de tabela inválidos/injeção (o nome vem do código, mas
    validar mantém o contrato explícito)."""
    if not nome.isidentifier():
        raise ValueError(f"nome de tabela inválido: {nome!r}")

In [4]:
# teste

casos = [
    "clientes",
    "tabela1",
    "123tabela",
    "minha tabela",
    "clientes!",
    "cliente ",
    " ",
]

for nome in casos:
    try:
        _validar_identificador(nome)
        print(f"{nome!r} -> válido")
    except ValueError as e:                     # guarda ValueError em "e" e faz print do nome e da mensagem de erro
        print(f"{nome!r} -> inválido: ({e})")

'clientes' -> válido
'tabela1' -> válido
'123tabela' -> inválido: (nome de tabela inválido: '123tabela')
'minha tabela' -> inválido: (nome de tabela inválido: 'minha tabela')
'clientes!' -> inválido: (nome de tabela inválido: 'clientes!')
'cliente ' -> inválido: (nome de tabela inválido: 'cliente ')
' ' -> inválido: (nome de tabela inválido: ' ')


In [5]:
def baixar_precos(
    ativos: Sequence[str],
    inicio: str,
    fim: str | None = None,
    frequencia: str = "1mo",
) -> pd.DataFrame:
    """Baixa preços de fechamento (ajustado) do Yahoo Finance. (F1)

    Parameters
    ----------
    ativos : sequência de tickers, ex.: ``["^BVSP"]``.
    inicio : data inicial ``AAAA-MM-DD``.
    fim : data final ``AAAA-MM-DD`` (``None`` => hoje).
    frequencia : intervalo do yfinance (``"1mo"`` = mensal).

    Returns
    -------
    DataFrame com a coluna ``data`` (``AAAA-MM``) na primeira posição e uma
    coluna por ticker (preço de fechamento ajustado).

    Notes
    -----
    Usa a *chart API* pública do Yahoo via ``urllib`` (stdlib) — **sem**
    ``yfinance``/``curl_cffi``, que sofriam de erro de certificado SSL no
    Windows. Imports tardios mantêm os testes de banco/retorno offline (NF4).
    """
    import json
    from datetime import datetime, timezone
    from urllib.parse import quote
    from urllib.request import Request, urlopen

    def _epoch(d: str) -> int:
        return int(pd.Timestamp(d, tz="UTC").timestamp())

    p1 = _epoch(inicio)
    p2 = _epoch(fim or datetime.now(timezone.utc).strftime("%Y-%m-%d"))

    series: dict[str, pd.Series] = {}
    for tk in ativos:
        url = (f"https://query1.finance.yahoo.com/v8/finance/chart/{quote(tk)}"
               f"?period1={p1}&period2={p2}&interval={frequencia}") 
        req = Request(url, headers={"User-Agent": "Mozilla/5.0"}) 
        with urlopen(req, timeout=30) as resp: 
            payload = json.load(resp) 
        resultado = (payload.get("chart") or {}).get("result") 
        if not resultado:
            raise ValueError(f"Yahoo não retornou dados para {tk!r}.")
        res = resultado[0] 
        ts = res.get("timestamp") or [] 
        close = res["indicators"]["quote"][0].get("close") or [] 
        meses = [datetime.fromtimestamp(t, tz=timezone.utc).strftime("%Y-%m") for t in ts] 
        series[tk] = pd.Series(close, index=meses, name=tk) 

    df = pd.DataFrame(series).dropna(how="any") 
    return df.reset_index().rename(columns={"index": "data"})

In [ ]:
def baixar_cdi_bcb(inicio: str, fim: str | None = None) -> pd.DataFrame:
    """Baixa o CDI mensal da API SGS do Banco Central. (F1)

    Série 4391 = 'Taxa de juros - CDI acumulada no mês' (% a.m.). Devolve um
    DataFrame com ``data`` (AAAA-MM) e ``cdi`` (decimal por mês, ex.: 0.0034).
    Fonte oficial, gratuita e sem cadastro; requer internet.

    Notes
    -----
    Imports tardios (``json``/``urllib``, stdlib) e nenhuma dependência nova.
    """
    import json
    from datetime import date as _date
    from urllib.request import urlopen

    di = pd.to_datetime(inicio).strftime("%d/%m/%Y")
    dfim = pd.to_datetime(fim or _date.today().isoformat()).strftime("%d/%m/%Y")
    url = ("https://api.bcb.gov.br/dados/serie/bcdata.sgs.4391/dados"
           f"?formato=json&dataInicial={di}&dataFinal={dfim}")
    with urlopen(url, timeout=30) as resp:
        dados = json.load(resp) 
    if not dados:
        raise ValueError("BCB não retornou CDI para o período pedido.")
    df = pd.DataFrame(dados)
    df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y").dt.strftime("%Y-%m") 
    df["cdi"] = df["valor"].astype(float) / 100.0   
    return df[["data", "cdi"]]

In [7]:
# teste

df = baixar_cdi_bcb(inicio="2020-01-01")

print(df)

       data     cdi
0   2020-01  0.0038
1   2020-02  0.0029
2   2020-03  0.0034
3   2020-04  0.0028
4   2020-05  0.0024
..      ...     ...
74  2026-03  0.0121
75  2026-04  0.0109
76  2026-05  0.0107
77  2026-06  0.0112
78  2026-07  0.0074

[79 rows x 2 columns]


In [8]:
def calcular_retornos(precos: pd.DataFrame, coluna_data: str = "data") -> pd.DataFrame:
    """Converte preços em retornos simples mensais. (F1)

    A primeira observação é descartada (não há retorno anterior), portanto
    ``T_efetivo = T − 1``. Todas as colunas que não sejam ``coluna_data`` são
    tratadas como séries de preço.

    Notes
    -----
    A taxa CDI, por **já ser** um retorno (não um preço), não passa por aqui:
    ela entra direto na coluna ``cdi`` da tabela ``retornos``.
    """
    if coluna_data not in precos.columns:
        raise KeyError(f"coluna '{coluna_data}' ausente em precos.")

    datas = precos[coluna_data].iloc[1:].to_numpy()
    numericas = precos.drop(columns=[coluna_data])
    ret = numericas.pct_change(fill_method=None).iloc[1:].reset_index(drop=True)
    ret.insert(0, coluna_data, datas)
    return ret

In [9]:
# teste

precos = pd.DataFrame({
    "data": ["2024-01", "2024-02", "2024-03", "2024-04"],
    "PETR4": [100, 110, 105, 120],
    "VALE3": [50, 55, 60, 58]
})

print(precos)

retornos = calcular_retornos(precos, coluna_data="data")
print(retornos)

      data  PETR4  VALE3
0  2024-01    100     50
1  2024-02    110     55
2  2024-03    105     60
3  2024-04    120     58
      data     PETR4     VALE3
0  2024-02  0.100000  0.100000
1  2024-03 -0.045455  0.090909
2  2024-04  0.142857 -0.033333


In [10]:
def gravar_sqlite(
    df: pd.DataFrame,
    db_path: str,
    tabela: str,
    if_exists: str = "replace",
) -> None:
    """Grava um DataFrame numa tabela do banco SQLite. (F1, NF6)

    Usa ``contextlib.closing`` porque ``with sqlite3.connect(...)`` apenas
    gerencia a transação — **não fecha** a conexão; sem fechar, o arquivo do
    banco fica travado no Windows (NF3).
    """
    _validar_identificador(tabela)
    with closing(sqlite3.connect(db_path)) as con:
        df.to_sql(tabela, con, if_exists=if_exists, index=False)
        con.commit()

In [ ]:
# teste

df = pd.DataFrame({
    "nome": ["João", "Maria"],
    "idade": [20, 25]
})

print(df)

db_path = "C:\\Users\\MURILO\\Desktop\\TCC\\Ambiente de Desenvolvimento\\teste.db"

tabela = "clientes" 

gravar_sqlite(df, db_path, tabela, if_exists="replace") 


    nome  idade
0   João     20
1  Maria     25


In [17]:
def ler_sqlite(db_path: str, tabela: str) -> pd.DataFrame:
    """Lê uma tabela do banco SQLite para um DataFrame. (F1, NF6)"""
    _validar_identificador(tabela)
    with closing(sqlite3.connect(db_path)) as con:
        return pd.read_sql(f"SELECT * FROM {tabela}", con)


In [18]:
# teste

ler_sqlite(db_path, tabela)

,nome,idade
0,João,20
1,Maria,25


**Teste** — retornos, round-trip SQLite e downloads reais (protegidos).

In [ ]:
import pandas as pd, os, tempfile
precos = pd.DataFrame({'data': ['2000-01','2000-02','2000-03','2000-04'], 
                       'ibov': [100.,102.,99.,105.]})
print('calcular_retornos:'); print(calcular_retornos(precos))

esp = precos['ibov'].pct_change(fill_method=None).iloc[1:].reset_index(drop=True)
assert calcular_retornos(precos)['ibov'].round(10).tolist() == esp.round(10).tolist()
db = os.path.join(tempfile.gettempdir(), 'dev01.db'); gravar_sqlite(precos, db, 'ibovespa')
lido = ler_sqlite(db, 'ibovespa'); pd.testing.assert_frame_equal(precos.reset_index(drop=True), lido, check_dtype=False); os.remove(db)
print('round-trip SQLite: OK')

try:
    print(baixar_precos(['^BVSP'], '2023-01-01', '2023-04-01'))
    print(baixar_cdi_bcb('2023-01-01', '2023-04-01'))
except Exception as e:
    print('download offline:', type(e).__name__, e)
print('OK')

calcular_retornos:
      data      ibov
0  2000-02  0.020000
1  2000-03 -0.029412
2  2000-04  0.060606
round-trip SQLite: OK
      data   ibov
0  2000-01  100.0
1  2000-02  102.0
2  2000-03   99.0
3  2000-04  105.0
      data     ^BVSP
0  2023-01  113532.0
1  2023-02  104932.0
2  2023-03  101882.0
download offline: JSONDecodeError Expecting value: line 1 column 1 (char 0)
OK


✔ **Testado e aprovado — código movido para `app/dal.py`.**

Funções da DAL desenvolvidas e testadas.